# SeedSec Rwanda: Maize Seed Vigor Detection Training Notebook
This Jupyter notebook contains the fully corrected training script to train a **YOLOv8** model for Maize Seed Vigor Detection. It mounts Google Drive, downloads/verifies the Kaggle dataset, recursively matches all images and XML labels, maps classes dynamically to prevent out-of-bounds corruption errors, trains the model, plots verification metrics, and exports it to TFLite for offline Android app integration.

## Step 1: Mount Google Drive & Set Kaggle Credentials

In [ ]:
import os
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

## Step 2: Install Libraries & Verify Dataset Location

In [ ]:
# Install Ultralytics for YOLOv8 and matplotlib/pandas for plotting metrics
!pip install ultralytics matplotlib pandas seaborn -q

# Verify dataset directory contents
dataset_dir = "/content/drive/MyDrive/KaggleDatasets/seed-vigor-detection"
print("Checking contents of dataset directory:")
!ls -la "$dataset_dir"

## Step 3: Unzip Dataset (If not already unzipped)
Unzips the dataset archive directly to Google Drive if the directories aren't already extracted.

In [ ]:
import zipfile
import os

zip_path = "/content/drive/MyDrive/KaggleDatasets/seed-vigor-detection/seed-vigor-detection-rgb-image.zip"
extract_path = "/content/drive/MyDrive/KaggleDatasets/seed-vigor-detection"

# Check if representative unzipped folders exist
if not os.path.exists(os.path.join(extract_path, 'Annotations')):
    if os.path.exists(zip_path):
        print(f"Unzipping {zip_path} to {extract_path}...")
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(extract_path)
        print("Unzipping complete.")
    else:
        print(f"Zip file not found at {zip_path}.")
else:
    print(f"Dataset already unzipped in {extract_path}.")

## Step 4: Convert Annotations & Organize Dataset
This cell searches recursively for all images and XML labels, pairs them, and dynamically populates the class mapping dictionary to avoid corrupt/out-of-bounds class errors.

In [ ]:
import xml.etree.ElementTree as ET
import glob
import shutil
import random
from pathlib import Path

# Paths configuration
source_dir = Path("/content/drive/MyDrive/KaggleDatasets/seed-vigor-detection")
yolo_dataset_dir = Path("/content/seed_vigor_yolo_format")

# Define target subdirectories
for split in ["train", "val"]:
    (yolo_dataset_dir / "images" / split).mkdir(parents=True, exist_ok=True)
    (yolo_dataset_dir / "labels" / split).mkdir(parents=True, exist_ok=True)

# 1. Dynamically capture ALL XML files recursively
print("Searching recursively for all XML files...")
all_xml_files = sorted(glob.glob(str(source_dir / "**" / "*.xml"), recursive=True))

# 2. Gather all image files recursively
all_image_files = sorted(glob.glob(str(source_dir / "**" / "*.jpg"), recursive=True) +
                         glob.glob(str(source_dir / "**" / "*.png"), recursive=True) +
                         glob.glob(str(source_dir / "**" / "*.jpeg"), recursive=True))

print(f"Total Images discovered on disk: {len(all_image_files)}")
print(f"Total XML annotations discovered on disk: {len(all_xml_files)}")

# 3. Create a map of stems to XML paths for lookups
xml_map = {}
for x in all_xml_files:
    stem = Path(x).stem
    xml_map[stem] = Path(x)

# Match images with their corresponding XMLs
dataset_pairs = []
for img_path_str in all_image_files:
    img_path = Path(img_path_str)
    if img_path.stem in xml_map:
        dataset_pairs.append((img_path, xml_map[img_path.stem]))

print(f"Successfully matched and paired: {len(dataset_pairs)} image-annotation sets.")

# Classes mapping dictionary (will append any newly discovered classes dynamically)
class_mapping = {}
discovered_classes = set()

def convert_xml_to_yolo(xml_file, output_txt_path, img_width, img_height):
    try:
        tree = ET.parse(xml_file)
        root = tree.getroot()
        size = root.find("size")
        if size is not None:
            w = size.find("width")
            h = size.find("height")
            img_width = int(w.text) if w is not None and int(w.text) > 0 else img_width
            img_height = int(h.text) if h is not None and int(h.text) > 0 else img_height
            
        lines = []
        for obj in root.findall("object"):
            class_name = obj.find("name").text.strip().lower()
            discovered_classes.add(class_name)
            
            # Dynamically register class names
            if class_name not in class_mapping:
                class_mapping[class_name] = len(class_mapping)
            class_id = class_mapping[class_name]
            
            bbox = obj.find("bndbox")
            xmin = float(bbox.find("xmin").text)
            ymin = float(bbox.find("ymin").text)
            xmax = float(bbox.find("xmax").text)
            ymax = float(bbox.find("ymax").text)
            
            # Normalize relative to image dimensions
            x_center = ((xmin + xmax) / 2.0) / img_width
            y_center = ((ymin + ymax) / 2.0) / img_height
            width = (xmax - xmin) / img_width
            height = (ymax - ymin) / img_height
            
            lines.append(f"{class_id} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}")
            
        with open(output_txt_path, "w") as f:
            f.write("\n".join(lines))
    except Exception as e:
        print(f"Error processing XML {xml_file}: {e}")

# 4. Split and Copy
if len(dataset_pairs) > 0:
    random.seed(42)
    random.shuffle(dataset_pairs)
    
    # Restrict to 5000 matched pairs for faster training, or use len(dataset_pairs) for full run
    max_samples = min(5000, len(dataset_pairs))
    selected_pairs = dataset_pairs[:max_samples]
    
    split_idx = int(len(selected_pairs) * 0.8)
    train_pairs = selected_pairs[:split_idx]
    val_pairs = selected_pairs[split_idx:]
    
    def process_pairs(pairs, split_name):
        count = 0
        for img_path, xml_path in pairs:
            dest_img = yolo_dataset_dir / "images" / split_name / img_path.name
            dest_lbl = yolo_dataset_dir / "labels" / split_name / f"{img_path.stem}.txt"
            shutil.copy(str(img_path), str(dest_img))
            convert_xml_to_yolo(xml_path, dest_lbl, 640, 640)
            count += 1
        print(f"Processed {count} entries into {split_name} subset.")

    process_pairs(train_pairs, "train")
    process_pairs(val_pairs, "val")
    print("Discovered classes in annotations:", list(discovered_classes))
    print("Final Class Mapping:", class_mapping)
else:
    print("No pairs found. Please check dataset path structures.")

## Step 5: Generate `dataset.yaml` Configuration
Creates the dynamic dataset configurations matching our discovered classes map.

In [ ]:
import yaml

classes_list = sorted(class_mapping.items(), key=lambda x: x[1])
classes_names = {idx: name for name, idx in classes_list}

yaml_content = {
    "path": "/content/seed_vigor_yolo_format",
    "train": "images/train",
    "val": "images/val",
    "names": classes_names
}

yaml_file_path = "/content/seed_vigor_yolo_format/dataset.yaml"
with open(yaml_file_path, "w") as f:
    yaml.dump(yaml_content, f, default_flow_style=False)

print("Created dataset.yaml content:")
with open(yaml_file_path, "r") as f:
    print(f.read())

## Step 6: Initialize and Train YOLOv8 Model
Uses transfer learning from pre-trained coco weights on YOLOv8n (lightweight nano model).

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

results = model.train(
    data="/content/seed_vigor_yolo_format/dataset.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,
    project="/content/drive/MyDrive/KaggleDatasets/seed-vigor-detection/runs",
    name="seed_vigor_detection_yolov8_final"
)

print("Training finished.")

## Step 7: Calculate Validation Metrics & F1-Score
Evaluates the final validation parameters including Precision, Recall, dynamic mAP scores, and the global F1-Score.

In [ ]:
best_weights_path = "/content/drive/MyDrive/KaggleDatasets/seed-vigor-detection/runs/seed_vigor_detection_yolov8_final/weights/best.pt"
best_model = YOLO(best_weights_path)

# Run validation
metrics = best_model.val()

# Extract Precision, Recall and mAP metrics
precision = metrics.box.mp
recall = metrics.box.mr
map50 = metrics.box.map50
map95 = metrics.box.map

# Compute Global Harmonic Mean F1-Score
if precision + recall > 0:
    f1_score = 2 * (precision * recall) / (precision + recall)
else:
    f1_score = 0.0

print("\n=================== VALIDATION PERFORMANCE METRICS ===================")
print(f"Precision (P):       {precision:.4f}")
print(f"Recall (R):          {recall:.4f}")
print(f"F1-Score:            {f1_score:.4f}")
print(f"mAP@50:              {map50:.4f}")
print(f"mAP@50-95:           {map95:.4f}")
print("======================================================================\n")

## Step 8: Plot Performance Curves & Diagrams
Displays training charts (Losses, Precision, Recall, mAP curves) and maps confusion matrices using matplotlib/seaborn.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

runs_dir = "/content/drive/MyDrive/KaggleDatasets/seed-vigor-detection/runs/seed_vigor_detection_yolov8_final"

# 1. Plot training metrics logs from csv
csv_path = os.path.join(runs_dir, "results.csv")
if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.strip()
    
    plt.figure(figsize=(16, 10))
    
    # Training and Validation Losses
    plt.subplot(2, 2, 1)
    plt.plot(df['epoch'], df['train/box_loss'], label='Train Box Loss', color='blue')
    plt.plot(df['epoch'], df['val/box_loss'], label='Val Box Loss', color='orange')
    plt.title('Bounding Box Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    
    plt.subplot(2, 2, 2)
    plt.plot(df['epoch'], df['train/cls_loss'], label='Train Class Loss', color='blue')
    plt.plot(df['epoch'], df['val/cls_loss'], label='Val Class Loss', color='orange')
    plt.title('Classification Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    
    # mAP Scores
    plt.subplot(2, 2, 3)
    plt.plot(df['epoch'], df['metrics/precision(B)'], label='Precision', color='green')
    plt.plot(df['epoch'], df['metrics/recall(B)'], label='Recall', color='red')
    plt.title('Precision & Recall')
    plt.xlabel('Epochs')
    plt.ylabel('Score')
    plt.legend()

    plt.subplot(2, 2, 4)
    plt.plot(df['epoch'], df['metrics/mAP50(B)'], label='mAP@50', color='purple')
    plt.plot(df['epoch'], df['metrics/mAP50-95(B)'], label='mAP@50-95', color='black')
    plt.title('Mean Average Precision (mAP)')
    plt.xlabel('Epochs')
    plt.ylabel('Score')
    plt.legend()
    
    plt.tight_layout()
    plt.show()
else:
    print("results.csv metrics file not found to plot curves.")

# 2. Render YOLOv8 generated Confusion Matrix and F1 Curve plots
for img_file in ["confusion_matrix.png", "F1_curve.png", "PR_curve.png"]:
    img_path = os.path.join(runs_dir, img_file)
    if os.path.exists(img_path):
        plt.figure(figsize=(8, 8))
        img = Image.open(img_path)
        plt.imshow(img)
        plt.title(img_file)
        plt.axis('off')
        plt.show()
    else:
        print(f"{img_file} plot file not found.")

## Step 9: Export Model to TFLite
Compiles the best weights checkpoint down to a FP16-quantized `.tflite` package ready to run on the mobile edge.

In [ ]:
print("Exporting best model to TFLite (Half-Precision)...")
tflite_path = best_model.export(format="tflite", int8=False, half=True)
print(f"TFLite model exported to: {tflite_path}")

# Copy to drive root
import shutil
shutil.copy(tflite_path, "/content/drive/MyDrive/KaggleDatasets/seed-vigor-detection/best_seed_vigor_fp16.tflite")
print("Model saved directly in Drive folder!")